<a href="https://colab.research.google.com/github/takumi-maker/dmd/blob/main/momentum_factor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import math
import statistics
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

In [37]:
df_1 = pd.read_csv("債券複利データセット3.csv")
df_1 = df_1.drop(df_1.columns[0], axis=1)
columns_len = len(df_1.columns)
length = len(df_1)
df_1
y_df = df_1["GJGC20"]
X_df = df_1.drop("GJGC20", axis=1)
y_df

,GJGC20
0,0.414
1,0.405
2,0.410
3,0.414
4,0.438
...,...
1107,2.525
1108,2.585
1109,2.568
1110,2.546


In [38]:
df_2 = pd.read_csv("債券複利データセット3.csv")
df_2 = df_2.drop(df_2.columns[0], axis=1)
columns_len = len(df_2.columns)
length = len(df_2)
df_2
y_df2 = df_2["GJGC30"]
X_df2 = df_2.drop("GJGC30", axis=1)
y_df2

,GJGC30
0,0.649
1,0.635
2,0.643
3,0.649
4,0.658
...,...
1107,3.022
1108,3.061
1109,3.034
1110,3.006


In [39]:
y_df = pd.concat([y_df, y_df2], axis=1)
y_df["select_spread"] = y_df["GJGC30"] - y_df["GJGC20"]
y_df = y_df.drop('GJGC20',axis=1)
y_df = y_df.drop('GJGC30',axis=1)

In [40]:
y_df

,select_spread
0,0.235
1,0.230
2,0.233
3,0.235
4,0.220
...,...
1107,0.497
1108,0.476
1109,0.466
1110,0.460


In [ ]:
from sklearn.linear_model import LinearRegression
import statsmodels
import statsmodels.api
import matplotlib.pyplot as plt
from sklearn import preprocessing
import cvxpy as cp

magic_num = 150
estimate_num = 150

df_array = y_df.to_numpy()
df_array = np.squeeze(df_array)

for i in range(1,8):
  total_num = 100*i
  print(total_num)

  total_list = []
  y_list = []
  result_list = []
  predict_flag2 = 0
  for j in range(magic_num-2,len(df_array)):
    X_return_list = []
    for k in range(1,magic_num):
      X_return_list.append((df_array[j-k]-df_array[j-k-1])*1)
    y_return = (df_array[j]-df_array[j-1])*1
    y_list.append(y_return)
    X_list2 = list(reversed(X_return_list))
    total_list.append(X_list2)
  #print(len(total_list))

  for j in range(len(total_list)):
    if j <= len(total_list)-1-total_num:
      continue
    #print(j)
    X_array = np.squeeze(np.array(total_list[j]))
    y_array = y_list[j]
    estimate_list = []
    estimate_y_list = []
    for k in range(1,estimate_num):
      estimate_list.append(total_list[j-k])
      estimate_y_list.append(y_list[j-k])

    estimate_list = np.array(estimate_list)
    estimate_list = np.squeeze(estimate_list)
    estimate_y_list = np.array(estimate_y_list)
    #print(estimate_list)
    #print(estimate_y_list)


    #alpha = np.array([0,0.1,0.2,0.3,0.4,0.5])
    #print(estimate_list.shape)
    #print(estimate_y_list.shape)
    k= estimate_list.shape[1]
    beta = cp.Variable(k,nonneg=True)
    constraints =[beta[i] <= beta[i+1]*0.9 for i in range(k-1)]
    #error = cp.norm2(estimate_y_list - estimate_list @ beta)
    #objective =  cp.sum_squares(estimate_y_list - estimate_list @ beta)
    objective =  cp.norm2(estimate_y_list - estimate_list @ beta)
    #objective = cp.Minimize(error)


    #prob = cp.Problem(objective, constraints)
    #prob = cp.Problem(cp.Minimize(objective), constraints)
    prob = cp.Problem(cp.Minimize(objective))
    prob.solve()
    #print("status:", prob.status)
    #print("optimal value", prob.value)
    #print("optimal var", beta.value)

    predict = np.dot(X_array,beta.value)
    #print(predict)
    #print(y_array)

    if predict >= 0.02: #5_30は0.05で-1
      predict_flag = -1
    elif predict <-0.02:
      predict_flag = 1
    else:
      predict_flag = predict_flag2

    if predict_flag*predict_flag2 >=0:
      result_r = y_array*predict_flag
      result_list.append(result_r)
      predict_flag2 = predict_flag
    else:
      result_r = y_array*predict_flag-0.005
      result_list.append(result_r)
      predict_flag2 = predict_flag

  R_sum = sum(result_list)
  SR = np.mean(np.array(result_list))/np.std(np.array(result_list))
  print("SharpRatio:",SR)
  print(R_sum)




100
SharpRatio: 0.06675982424127186
0.11899999999999977
200


In [ ]:
#モメンタム順張り
from sklearn.linear_model import LinearRegression
import statsmodels
import statsmodels.api
import matplotlib.pyplot as plt
from sklearn import preprocessing
import cvxpy as cp

magic_num = 150
estimate_num = 150

df_array = y_df2.to_numpy()

for i in range(1,8):
  total_num = 100*i
  print(total_num)

  total_list = []
  y_list = []
  result_list = []
  predict_flag2 = 0
  for j in range(magic_num-2,len(df_array)):
    X_return_list = []
    for k in range(1,magic_num):
      X_return_list.append((df_array[j-k]-df_array[j-k-1])*1)
    y_return = (df_array[j]-df_array[j-1])*1
    y_list.append(y_return)
    X_list2 = list(reversed(X_return_list))
    total_list.append(X_list2)
  #print(len(total_list))

  for j in range(len(total_list)):
    if j <= len(total_list)-1-total_num:
      continue
    #print(j)
    X_array = np.squeeze(np.array(total_list[j]))
    y_array = y_list[j]
    estimate_list = []
    estimate_y_list = []
    for k in range(1,estimate_num):
      estimate_list.append(total_list[j-k])
      estimate_y_list.append(y_list[j-k])

    estimate_list = np.array(estimate_list)
    estimate_list = np.squeeze(estimate_list)
    estimate_y_list = np.array(estimate_y_list)
    #print(estimate_list)
    #print(estimate_y_list)


    #alpha = np.array([0,0.1,0.2,0.3,0.4,0.5])
    #print(estimate_list.shape)
    #print(estimate_y_list.shape)
    k= estimate_list.shape[1]
    beta = cp.Variable(k,nonneg=True)
    constraints =[beta[i] <= beta[i+1]*0.9 for i in range(k-1)]
    #error = cp.norm2(estimate_y_list - estimate_list @ beta)
    objective =  cp.sum_squares(estimate_y_list - estimate_list @ beta)
    #objective = cp.Minimize(error)


    #prob = cp.Problem(objective, constraints)
    prob = cp.Problem(cp.Minimize(objective), constraints)
    prob.solve()
    #print("status:", prob.status)
    #print("optimal value", prob.value)
    #print("optimal var", beta.value)

    predict = np.dot(X_array,beta.value)
    #print(predict)
    #print(y_array)

    if predict >= 0:
      predict_flag = 1
    else:
      predict_flag = -1

    if predict_flag*predict_flag2 >=0:
      result_r = y_array*predict_flag
      result_list.append(result_r)
      predict_flag2 = predict_flag
    else:
      result_r = y_array*predict_flag-0.005
      result_list.append(result_r)
      predict_flag2 = predict_flag

  R_sum = sum(result_list)
  SR = np.mean(np.array(result_list))/np.std(np.array(result_list))
  print("SharpRatio:",SR)
  print(R_sum)



100
SharpRatio: -0.15368225374519173
-0.4260000000000004
200
SharpRatio: -0.09936052547897195
-0.4400000000000006
300
SharpRatio: -0.04973188974207281
-0.3370000000000005
400
SharpRatio: -0.06844885874780146
-0.5670000000000006
500
SharpRatio: -0.07695921588912734
-0.7390000000000005
600
SharpRatio: -0.0856695866229294
-0.9170000000000004
700
SharpRatio: -0.0945971507182608
-1.1193999999999997


In [ ]:
#モメンタム逆張り
from sklearn.linear_model import LinearRegression
import statsmodels
import statsmodels.api
import matplotlib.pyplot as plt
from sklearn import preprocessing
import cvxpy as cp

magic_num = 30
estimate_num = 150

df_array = y_df2.to_numpy()

for i in range(1,8):
  total_num = 100*i
  print(total_num)

  total_list = []
  y_list = []
  result_list = []
  predict_flag2 = 0
  for j in range(magic_num-2,len(df_array)):
    X_return_list = []
    for k in range(1,magic_num):
      X_return_list.append((df_array[j-k]-df_array[j-k-1])*1)
    y_return = (df_array[j]-df_array[j-1])*1
    y_list.append(y_return)
    X_list2 = list(reversed(X_return_list))
    total_list.append(X_list2)
  #print(len(total_list))

  for j in range(len(total_list)):
    if j <= len(total_list)-1-total_num:
      continue
    #print(j)
    X_array = np.squeeze(np.array(total_list[j]))
    y_array = y_list[j]
    estimate_list = []
    estimate_y_list = []
    for k in range(1,estimate_num):
      estimate_list.append(total_list[j-k])
      estimate_y_list.append(y_list[j-k])

    estimate_list = np.array(estimate_list)
    estimate_list = np.squeeze(estimate_list)
    estimate_y_list = np.array(estimate_y_list)
    #print(estimate_list)
    #print(estimate_y_list)


    #alpha = np.array([0,0.1,0.2,0.3,0.4,0.5])
    #print(estimate_list.shape)
    #print(estimate_y_list.shape)
    k= estimate_list.shape[1]
    beta = cp.Variable(k,nonneg=True)
    constraints =[beta[i] <= beta[i+1]*0.9 for i in range(k-1)]
    #error = cp.norm2(estimate_y_list - estimate_list @ beta)
    objective =  cp.sum_squares(estimate_y_list - estimate_list @ beta)
    #objective = cp.Minimize(error)


    #prob = cp.Problem(objective, constraints)
    prob = cp.Problem(cp.Minimize(objective), constraints)
    prob.solve()
    #print("status:", prob.status)
    #print("optimal value", prob.value)
    #print("optimal var", beta.value)

    predict = np.dot(X_array,beta.value)
    #print(predict)
    #print(y_array)

    if predict/abs(df_array[j-1]) >= 0.00:
      predict_flag = -1
    elif predict/abs(df_array[j-1]) <-0.00:
      predict_flag = 1
    else:
      predict_flag = predict_flag2

    if predict_flag*predict_flag2 >=0:
      result_r = y_array*predict_flag
      result_list.append(result_r)
      predict_flag2 = predict_flag
    else:
      result_r = y_array*predict_flag-0.00
      result_list.append(result_r)
      predict_flag2 = predict_flag

  R_sum = sum(result_list)
  SR = np.mean(np.array(result_list))/np.std(np.array(result_list))
  print("SharpRatio:",SR)
  print(R_sum)

100
SharpRatio: 0.0773900546186061
0.2140000000000004
200
SharpRatio: 0.010909852822112024
0.04800000000000049
300
SharpRatio: -0.040452255554498565
-0.2749999999999998
400
SharpRatio: -0.03599420683866334
-0.2999999999999998
500
SharpRatio: -0.038177581210843185
-0.367
600
SharpRatio: -0.025553896631303606
-0.2739999999999999
700


KeyboardInterrupt: 

In [ ]:
!pip install cvxpy

In [ ]:
#モメンタム逆張り
from sklearn.linear_model import LinearRegression
import statsmodels
import statsmodels.api
import matplotlib.pyplot as plt
from sklearn import preprocessing
import cvxpy as cp

magic_num = 100
estimate_num = 150

df_array = y_df2.to_numpy()

for i in range(1,8):
  total_num = 100*i
  print(total_num)

  total_list = []
  y_list = []
  result_list = []
  predict_flag2 = 0
  for j in range(magic_num-2,len(df_array)):
    X_return_list = []
    for k in range(1,magic_num):
      X_return_list.append((df_array[j-k]-df_array[j-k-1])*1)
    y_return = (df_array[j]-df_array[j-1])*1
    y_list.append(y_return)
    X_list2 = list(reversed(X_return_list))
    total_list.append(X_list2)
  #print(len(total_list))

  for j in range(len(total_list)):
    if j <= len(total_list)-1-total_num:
      continue
    #print(j)
    X_array = np.squeeze(np.array(total_list[j]))
    y_array = y_list[j]
    estimate_list = []
    estimate_y_list = []
    for k in range(1,estimate_num):
      estimate_list.append(total_list[j-k])
      estimate_y_list.append(y_list[j-k])

    estimate_list = np.array(estimate_list)
    estimate_list = np.squeeze(estimate_list)
    estimate_y_list = np.array(estimate_y_list)
    #print(estimate_list)
    #print(estimate_y_list)


    #alpha = np.array([0,0.1,0.2,0.3,0.4,0.5])
    #print(estimate_list.shape)
    #print(estimate_y_list.shape)
    k= estimate_list.shape[1]
    beta = cp.Variable(k,nonneg=True)
    constraints =[beta[i] <= beta[i+1]*0.9 for i in range(k-1)]
    #error = cp.norm2(estimate_y_list - estimate_list @ beta)
    objective =  cp.sum_squares(estimate_y_list - estimate_list @ beta)
    #objective = cp.Minimize(error)


    #prob = cp.Problem(objective, constraints)
    prob = cp.Problem(cp.Minimize(objective), constraints)
    prob.solve()
    #print("status:", prob.status)
    #print("optimal value", prob.value)
    #print("optimal var", beta.value)

    predict = np.dot(X_array,beta.value)
    #print(predict)
    #print(y_array)

    if predict >= 0.00:
      predict_flag = -1
    elif predict <-0.00:
      predict_flag = 1
    else:
      predict_flag = predict_flag2

    if predict_flag*predict_flag2 >=0:
      result_r = y_array*predict_flag
      result_list.append(result_r)
      predict_flag2 = predict_flag
    else:
      result_r = y_array*predict_flag-0.00
      result_list.append(result_r)
      predict_flag2 = predict_flag

  R_sum = sum(result_list)
  SR = np.mean(np.array(result_list))/np.std(np.array(result_list))
  print("SharpRatio:",SR)
  print(R_sum)

100
SharpRatio: 0.06843678064531152
0.14500000000000002
200
SharpRatio: 0.03745259551087568
0.2150000000000003
300
SharpRatio: 0.08152088291821559
0.7050000000000004
400
SharpRatio: 0.06859795911515162
0.7620000000000003
500
SharpRatio: 0.06209768680654199
0.8610000000000002
600
SharpRatio: 0.06172639782858514
0.9949999999999997
700
SharpRatio: 0.05754283520091026
1.0439999999999987
